# Lab 3: Test MCP Server

배포된 MCP 서버의 13개 도구를 테스트합니다. AgentCore Gateway의 MCP endpoint를 통해 호출합니다.

> ⚠️ 첫 호출 시 Livy 세션 생성에 1-2분 소요됩니다. 이후 호출은 수 초 내 응답합니다.

## Step 1: Setup MCP Client

In [ ]:
import boto3
import json
import time

REGION = "us-west-2"
FUNCTION_NAME = "fhir-mcp-server"
lambda_client = boto3.client("lambda", region_name=REGION)

def invoke_tool(tool_name: str, arguments: dict = {}) -> dict:
    """MCP 도구를 Lambda 직접 호출로 테스트합니다."""
    start = time.time()
    resp = lambda_client.invoke(
        FunctionName=FUNCTION_NAME,
        Payload=json.dumps({"toolName": tool_name, "arguments": arguments})
    )
    result = json.loads(resp["Payload"].read())
    elapsed = time.time() - start
    print(f"⏱ {elapsed:.1f}s | {tool_name}")
    if result.get("status") == "error":
        print(f"❌ Error: {result.get('message')}")
    else:
        print(json.dumps(result.get("result", result), indent=2, ensure_ascii=False)[:3000])
    return result

print("Ready!")

## Test 1: Schema Discovery

데이터 레이크의 테이블 구조를 탐색합니다.

In [ ]:
# 전체 테이블 목록
invoke_tool("list_tables")

In [ ]:
# 임상 도메인 테이블만
invoke_tool("list_tables", {"domain": "clinical"})

In [ ]:
# patient_registry 스키마 확인
invoke_tool("get_table_schema", {"table_name": "patient_registry"})

In [ ]:
# clinical_encounter 테이블 관계 확인
invoke_tool("get_table_relationships", {"table_name": "clinical_encounter"})

## Test 2: Patient Search

환자를 검색하고 상세 정보를 조회합니다.

In [ ]:
# 당뇨 환자 검색
result = invoke_tool("search_patients", {"condition_code": "diabetes", "gender": "female"})

In [ ]:
# 검색 결과에서 첫 번째 환자 ID로 종합 요약 조회
patients = result.get("result", [])
if patients:
    pid = patients[0]["resource_id"] if isinstance(patients[0], dict) else None
    if pid:
        print(f"\n--- Patient Summary for {pid} ---")
        invoke_tool("get_patient_summary", {"patient_id": pid})

## Test 3: Clinical Data

진료 이력, 관찰 데이터, 투약, 진단 이력을 조회합니다.

> 위 테스트에서 얻은 `pid`를 사용합니다. 없으면 아래 셀에서 직접 입력하세요.

In [ ]:
# pid = "직접-입력-환자-ID"  # 필요시 주석 해제

# 진료 이력
invoke_tool("get_encounter_history", {"patient_id": pid})

In [ ]:
# 관찰 데이터 (혈당)
invoke_tool("get_clinical_observations", {"patient_id": pid, "observation_code": "glucose"})

In [ ]:
# 현재 투약 중인 약물
invoke_tool("get_medications", {"patient_id": pid, "active_only": True})

In [ ]:
# 진단 이력
invoke_tool("get_diagnosis_history", {"patient_id": pid})

## Test 4: Financial & Analytics

In [ ]:
# 청구 요약
invoke_tool("get_claim_summary", {"patient_id": pid})

In [ ]:
# 케어 갭 분석
invoke_tool("detect_care_gaps", {"patient_id": pid})

In [ ]:
# 인구 건강 지표 - 당뇨 60대
invoke_tool("get_population_health_metrics", {"condition_code": "diabetes", "age_group": "60-69"})

## Test 5: Custom Query (Text-to-SQL)

자유 SQL 쿼리를 실행합니다.

In [ ]:
# 가장 많이 처방된 약물 Top 10
invoke_tool("run_custom_query", {
    "query": """
    SELECT medication_code_display, COUNT(*) as cnt
    FROM s3tablescatalog.`fhir-bucket.data`.medication_request
    GROUP BY medication_code_display
    ORDER BY cnt DESC
    LIMIT 10
    """
})

In [ ]:
# DML 차단 테스트 (에러가 나야 정상)
invoke_tool("run_custom_query", {"query": "DROP TABLE patient_registry"})

## ✅ Test Summary

| Category | Tools | Status |
|----------|-------|--------|
| Schema Discovery | list_tables, get_table_schema, get_table_relationships | |
| Patient | search_patients, get_patient_summary | |
| Clinical | get_encounter_history, get_clinical_observations, get_medications, get_diagnosis_history | |
| Financial | get_claim_summary | |
| Analytics | detect_care_gaps, get_population_health_metrics | |
| Query | run_custom_query (SELECT ✅, DML ❌) | |